# 0. Load imports 

In [57]:
import pandas as pd
import numpy as np
import re
import gdown #needed to get data from google drive

import matplotlib.pyplot as plt
import seaborn as sns

## print multiple things from same cell
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

## load data of post 2000 UN voting data
gdown.download(id="1aHnVYItfvHossY7LpmnsQimvx1wNko1f", output="2026_02_06_ga_voting.csv", quiet=False)
df = pd.read_csv("2026_02_06_ga_voting.csv")

print(df.shape)
print(df.head())


Downloading...
From (original): https://drive.google.com/uc?id=1aHnVYItfvHossY7LpmnsQimvx1wNko1f
From (redirected): https://drive.google.com/uc?id=1aHnVYItfvHossY7LpmnsQimvx1wNko1f&confirm=t&uuid=78b76c76-3cc1-46b4-a8e3-888a594f873a
To: /Users/charlesphillips/Documents/GitHub/qss20-final-project/code/2026_02_06_ga_voting.csv
100%|████████████████████████████████████████| 364M/364M [00:09<00:00, 38.0MB/s]


'2026_02_06_ga_voting.csv'

/var/folders/cs/x74m_3cj4xv9_rl_f_d37ts40000gn/T/ipykernel_21206/2184471.py:15: DtypeWarning: Columns (5,13) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("2026_02_06_ga_voting.csv")


(947434, 20)
   undl_id ms_code      ms_name ms_vote        date session   resolution  \
0   507407     AFG  AFGHANISTAN       Y  2003-12-03      58  A/RES/58/20   
1   507407     ALB      ALBANIA       Y  2003-12-03      58  A/RES/58/20   
2   507407     DZA      ALGERIA       Y  2003-12-03      58  A/RES/58/20   
3   507407     AND      ANDORRA       Y  2003-12-03      58  A/RES/58/20   
4   507407     AGO       ANGOLA       X  2003-12-03      58  A/RES/58/20   

                       draft committee_report     meeting  \
0  A/58/L.25|A/58/L.25/Add.1              NaN  A/58/PV.68   
1  A/58/L.25|A/58/L.25/Add.1              NaN  A/58/PV.68   
2  A/58/L.25|A/58/L.25/Add.1              NaN  A/58/PV.68   
3  A/58/L.25|A/58/L.25/Add.1              NaN  A/58/PV.68   
4  A/58/L.25|A/58/L.25/Add.1              NaN  A/58/PV.68   

                                               title            agenda_title  \
0  Special information programme on the question ...  Question of Palestine.   
1  

# 1. Filter data to post 2000

I considered doing this by session, but sessions run Sept-Sept, so it would be harder to isolate Trump's changes.

In [58]:
## Filter to post 2000
df['year'] = pd.to_datetime(df['date']).dt.year
df_post2000 = df[df['year'] >= 2000].copy()

# 2. Exploring the data

In [59]:
## How many votes per year
print(df_post2000['year'].min())
print(df_post2000['year'].value_counts().sort_index())

2000
year
2000    12851
2001    12852
2002    14508
2003    14707
2004    13943
2005    14325
2006    19390
2007    15168
2008    14976
2009    13056
2010    13824
2011    13312
2012    14282
2013    12352
2014    15440
2015    15054
2016    15633
2017    18142
2018    20651
2019    19300
2020    19300
2021    16598
2022    17177
2023    16984
2024    18335
2025    37056
Name: count, dtype: int64


In [60]:
## How many resolutions per year and total number of resolutions
df_post2000.groupby('year')['resolution'].nunique().sort_index()
print(df_post2000['resolution'].nunique())

year
2000     68
2001     68
2002     76
2003     77
2004     73
2005     75
2006    101
2007     79
2008     78
2009     68
2010     72
2011     69
2012     74
2013     64
2014     80
2015     78
2016     81
2017     94
2018    107
2019    100
2020    100
2021     86
2022     89
2023     88
2024     95
2025    192
Name: resolution, dtype: int64

2232


In [61]:
## Number of unique countries
print(df_post2000['ms_name'].value_counts())
print(df_post2000['ms_name'].nunique())

ms_name
AFGHANISTAN                     2232
MYANMAR                         2232
NAURU                           2232
NEPAL                           2232
NEW ZEALAND                     2232
                                ... 
NETHERLANDS (KINGDOM OF THE)     365
VENEZUELA                        297
SERBIA AND MONTENEGRO            227
YUGOSLAVIA                       212
TÜRKÝYE                           70
Name: count, Length: 206, dtype: int64
206


In [62]:
## What are the voting options
print(df_post2000['ms_vote'].unique())
print(df_post2000['ms_vote'].value_counts())

['Y' 'X' 'A' 'N']
ms_vote
Y    318454
A     44186
X     39503
N     27073
Name: count, dtype: int64


In [63]:
## are there any NAs?
print(df_post2000['ms_vote'].isna().sum())

0


# 3. Cleaning the country titles to resolve differing names

I noticed how some countries had multiple names (e.g. Turkey). However, they have the same country code. So I standardized the country names so that each code has just 1. 

In [64]:
print(sorted(df_post2000['ms_name'].unique()))

['AFGHANISTAN', 'ALBANIA', 'ALGERIA', 'ANDORRA', 'ANGOLA', 'ANTIGUA AND BARBUDA', 'ARGENTINA', 'ARMENIA', 'AUSTRALIA', 'AUSTRIA', 'AZERBAIJAN', 'BAHAMAS', 'BAHRAIN', 'BANGLADESH', 'BARBADOS', 'BELARUS', 'BELGIUM', 'BELIZE', 'BENIN', 'BHUTAN', 'BOLIVIA', 'BOLIVIA (PLURINATIONAL STATE OF)', 'BOSNIA AND HERZEGOVINA', 'BOTSWANA', 'BRAZIL', 'BRUNEI DARUSSALAM', 'BULGARIA', 'BURKINA FASO', 'BURUNDI', 'CABO VERDE', 'CAMBODIA', 'CAMEROON', 'CANADA', 'CAPE VERDE', 'CENTRAL AFRICAN REPUBLIC', 'CHAD', 'CHILE', 'CHINA', 'COLOMBIA', 'COMOROS', 'CONGO', 'COSTA RICA', "COTE D'IVOIRE", 'CROATIA', 'CUBA', 'CYPRUS', 'CZECH REPUBLIC', 'CZECHIA', "CÔTE D'IVOIRE", "DEMOCRATIC PEOPLE'S REPUBLIC OF KOREA", 'DEMOCRATIC REPUBLIC OF THE CONGO', 'DENMARK', 'DJIBOUTI', 'DOMINICA', 'DOMINICAN REPUBLIC', 'ECUADOR', 'EGYPT', 'EL SALVADOR', 'EQUATORIAL GUINEA', 'ERITREA', 'ESTONIA', 'ESWATINI', 'ETHIOPIA', 'FIJI', 'FINLAND', 'FRANCE', 'GABON', 'GAMBIA', 'GEORGIA', 'GERMANY', 'GHANA', 'GREECE', 'GRENADA', 'GUATEMALA',

In [65]:
# for each country code, what country names correspond to them. filters to those which have more than 1.
# output is the code and the multiple names associated w that code

code_name_check = df_post2000.groupby('ms_code')['ms_name'].nunique()
problem_codes = code_name_check[code_name_check > 1].index
for code in problem_codes:
    print(code, df_post2000[df_post2000['ms_code'] == code]['ms_name'].unique())

BOL ['BOLIVIA' 'BOLIVIA (PLURINATIONAL STATE OF)']
CIV ["CÔTE D'IVOIRE" "COTE D'IVOIRE"]
CPV ['CAPE VERDE' 'CABO VERDE']
CZE ['CZECH REPUBLIC' 'CZECHIA']
LBY ['LIBYAN ARAB JAMAHIRIYA' 'LIBYA']
MKD ['THE FORMER YUGOSLAV REPUBLIC OF MACEDONIA' 'NORTH MACEDONIA']
NLD ['NETHERLANDS' 'NETHERLANDS (KINGDOM OF THE)']
SWZ ['SWAZILAND' 'ESWATINI']
TUR ['TURKEY' 'TÜRKİYE' 'TÜRKÝYE']
VEN ['VENEZUELA' 'VENEZUELA (BOLIVARIAN REPUBLIC OF)']


In [66]:
## standardize the names
code_to_name = {
    'BOL': 'BOLIVIA',
    'CIV': "COTE D'IVOIRE",
    'CPV': 'CABO VERDE',
    'CZE': 'CZECHIA',
    'LBY': 'LIBYA',
    'MKD': 'NORTH MACEDONIA',
    'NLD': 'NETHERLANDS',
    'SWZ': 'ESWATINI',
    'TUR': 'TURKEY',
    'VEN': 'VENEZUELA'
}
df_post2000['ms_name'] = df_post2000['ms_code'].map(code_to_name).fillna(df_post2000['ms_name'])

print(df_post2000['ms_name'].nunique())

195


# 4. Cleaning the voting options

I found that there are 4 voting options: Yes, No, Abstain and Absent. I decided to remove absent, but usually abstaining has meaning . Therefore, I will keep that in as a middle ground value of 0.5, while No is 0 and Yes is 1.

In [67]:
df_post2000 = df_post2000[df_post2000['ms_vote'] != 'X']
print(df_post2000['ms_vote'].value_counts())

ms_vote
Y    318454
A     44186
N     27073
Name: count, dtype: int64


In [68]:
vote_score = {'Y': 1, 'A': 0.5, 'N': 0}
df_post2000['ms_vote_score'] = df_post2000['ms_vote'].map(vote_score)

# 5. Alignment Scores for the US and China

In [69]:
# Isolating what the US and China voted in all these resolutions, and adding that as a column to all countries and their votes
us_votes = df_post2000[df_post2000['ms_code'] == 'USA'][['resolution', 'ms_vote_score']].rename(columns={'ms_vote_score': 'us_vote_score'})
china_votes = df_post2000[df_post2000['ms_code'] == 'CHN'][['resolution', 'ms_vote_score']].rename(columns={'ms_vote_score': 'china_vote_score'})

#merge onto full dataset 
df_post2000 = df_post2000.merge(us_votes, on='resolution', how='left')
df_post2000 = df_post2000.merge(china_votes, on='resolution', how='left')

# 6. Filter to votes where the US and China are misaligned

In [72]:
df_post2000 = df_post2000[df_post2000['us_vote_score'] != df_post2000['china_vote_score']]
df_post2000

,undl_id,ms_code,ms_name,ms_vote,date,session,resolution,draft,committee_report,meeting,...,total_yes,total_no,total_abstentions,total_non_voting,total_ms,undl_link,year,ms_vote_score,us_vote_score,china_vote_score
0,507407,AFG,AFGHANISTAN,Y,2003-12-03,58,A/RES/58/20,A/58/L.25|A/58/L.25/Add.1,NaN,A/58/PV.68,...,159.0,6.0,6.0,20.0,191.0,https://digitallibrary.un.org/record/507407,2003,1.0,0.0,1.0
1,507407,ALB,ALBANIA,Y,2003-12-03,58,A/RES/58/20,A/58/L.25|A/58/L.25/Add.1,NaN,A/58/PV.68,...,159.0,6.0,6.0,20.0,191.0,https://digitallibrary.un.org/record/507407,2003,1.0,0.0,1.0
2,507407,DZA,ALGERIA,Y,2003-12-03,58,A/RES/58/20,A/58/L.25|A/58/L.25/Add.1,NaN,A/58/PV.68,...,159.0,6.0,6.0,20.0,191.0,https://digitallibrary.un.org/record/507407,2003,1.0,0.0,1.0
3,507407,AND,ANDORRA,Y,2003-12-03,58,A/RES/58/20,A/58/L.25|A/58/L.25/Add.1,NaN,A/58/PV.68,...,159.0,6.0,6.0,20.0,191.0,https://digitallibrary.un.org/record/507407,2003,1.0,0.0,1.0
4,507407,ATG,ANTIGUA AND BARBUDA,Y,2003-12-03,58,A/RES/58/20,A/58/L.25|A/58/L.25/Add.1,NaN,A/58/PV.68,...,159.0,6.0,6.0,20.0,191.0,https://digitallibrary.un.org/record/507407,2003,1.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
389708,4088513,URY,URUGUAY,Y,2025-09-19,80,A/RES/80/1,A/80/L.2/Rev.1,NaN,A/80/PV.3,...,145.0,5.0,6.0,37.0,193.0,https://digitallibrary.un.org/record/4088513,2025,1.0,0.0,1.0
389709,4088513,UZB,UZBEKISTAN,Y,2025-09-19,80,A/RES/80/1,A/80/L.2/Rev.1,NaN,A/80/PV.3,...,145.0,5.0,6.0,37.0,193.0,https://digitallibrary.un.org/record/4088513,2025,1.0,0.0,1.0
389710,4088513,VNM,VIET NAM,Y,2025-09-19,80,A/RES/80/1,A/80/L.2/Rev.1,NaN,A/80/PV.3,...,145.0,5.0,6.0,37.0,193.0,https://digitallibrary.un.org/record/4088513,2025,1.0,0.0,1.0
389711,4088513,YEM,YEMEN,Y,2025-09-19,80,A/RES/80/1,A/80/L.2/Rev.1,NaN,A/80/PV.3,...,145.0,5.0,6.0,37.0,193.0,https://digitallibrary.un.org/record/4088513,2025,1.0,0.0,1.0


year
2000     9331
2001     9489
2002    11288
2003    11946
2004    11857
2005    11978
2006    16478
2007    13037
2008    12397
2009    10764
2010    11354
2011    10186
2012    11887
2013    10187
2014    12319
2015    11810
2016    12405
2017    14335
2018    17085
2019    15436
2020    15928
2021    12710
2022    13073
2023    13069
2024    13896
2025    32005
Name: count, dtype: int64


# 7. For each year, alignment score for each country with the US/China

Perfect alignment = 1. To calculate the alignment w US/China, take the absolute value of the difference between their score and US score and remove it from 1.

In [73]:
df_post2000['align_us'] = 1 - abs(df_post2000['ms_vote_score'] - df_post2000['us_vote_score'])
df_post2000['align_china'] = 1 - abs(df_post2000['ms_vote_score'] - df_post2000['china_vote_score'])

# 8. Save this cleaned data

I saved it, but was too large for Github, so i dragged it into the Google Drive. 

In [82]:
#df_post2000.to_csv('../data/processed/df_clean.csv', index=False)